Ejecuta la siguiente celda e introduce la contraseña facilitada en el curso para desbloquear el cuaderno.


In [ ]:
%pip install -q git+https://github.com/dcrespons14/cam-estrategia-aula.git
from cam_estrategia import desbloquear_cuaderno
desbloquear_cuaderno()


# Modelo de carrera completo

En este cuaderno construiremos un **modelo de carrera** a partir de los Grandes Premios de México de 2023, 2024 y 2025.

Usaremos los modelos de **neumáticos** y **adelantamients** para construir una carrera completa.

Los datos vienen de [OpenF1](https://openf1.org), una API pública y gratuita.

## Inicialización

Importamos las librerías relevantes.

In [ ]:
# Librerías propias (ver src/race_analysis y src/race_models)
from race_analysis.ingestion.openf1 import get, load_race
from race_analysis.ingestion.laps import clean_data
from race_analysis.ingestion.race_history import real_race_result
from race_analysis.tyre_deg.pace import fit_pooled_race_pace
from race_analysis.tyre_deg.degradation_plots import plot_track_evolution
from race_analysis.overtaking_model.overtaking import find_overtakes, attack_situations, fit_logistic
from race_analysis.overtaking_model.overtaking_plots import plot_logistic
from race_models.f1.config import RaceConfig
from race_models.f1.models import base_pace_from_race_pace, tyre_deg_from_race_pace
from race_models.f1.engine.simulator import simulate_race, simulate_strategy
from race_models.f1.analysis.race_plots import (plot_base_pace, plot_tyre_model, plot_positions, plot_race_trace,
                                                plot_tyre_strategies, plot_time_lost)

# Librerías de Python
import numpy as np
import pandas as pd
from IPython.display import display

## 1. Descarga de datos

Descargamos y limpiamos las vueltas de las tres carreras y corregimos el efecto del combustible, igual que en los cuadernos de las clases 4.3 y 4.5.

In [ ]:
TRACK = "Mexico"
YEARS = [2023, 2024, 2025]

races = {year: load_race(year, TRACK) for year in YEARS}
laps = {year: clean_data(race) for year, race in races.items()}

FUEL_EFFECT = 0.05  # s por vuelta de combustible a bordo (estimación habitual en F1)
for frame in laps.values():
    frame["fuel_corrected"] = frame["lap_time"] - FUEL_EFFECT * frame["laps_remaining"]

for year, race in races.items():
    print(f"{year}: {race.total_laps} vueltas")

## 2. El modelo de neumáticos

Usamos el mismo modelo lineal que en los cuadernos de las clase 4.3.

In [ ]:
linear = fit_pooled_race_pace(pd.concat(laps.values(), ignore_index=True), model="lineal")
pace = linear[2025]

print(f"Error del modelo: {pace.rmse:.2f} s")
print("Diferencia con el medio (s):", {name: round(value, 2) for name, value in pace.compound_offset.items()})
print("Degradación (s/vuelta):     ", {name: round(value, 3) for name, value in pace.degradation.items()})

In [ ]:
tlas = {driver["driver_number"]: driver["name_acronym"] for driver in get("drivers", session_key=races[2025].session_key)}

base_pace = base_pace_from_race_pace(pace, tlas)
tyre_deg = tyre_deg_from_race_pace(pace)

plot_tyre_model(pace.tyre_curves(), max_life=45, title="México 2023-2025 · Modelo lineal");
plot_base_pace(base_pace, title="México 2025 · Ritmo base");

## 3. El modelo de adelantamiento

Repetimos el modelo del cuaderno de la clase 4.5.

In [ ]:
MAX_GAP = 1.5  # distancia máxima (s) con el coche de delante para considerar que hay un ataque

# Adelantamientos válidos (los que vienen de una situación de ataque) y situaciones de ataque
overtaking_data = {year: find_overtakes(race, laps[year], max_gap=MAX_GAP) for year, race in races.items()}
attacks = pd.concat([attack_situations(races[year], laps[year], linear[year], overtaking_data[year], max_gap=MAX_GAP)
                     for year in YEARS], ignore_index=True)
overtaking_model = fit_logistic(attacks["delta_pace"], attacks["overtake"])

print(f"Situaciones de ataque: {len(attacks)}, adelantamientos: {attacks['overtake'].sum()} ({attacks['overtake'].mean():.0%})")
print(f"β0 = {overtaking_model.intercept:.3f}, β1 = {overtaking_model.slope:.3f}")
print(f"Para un 50 %: {overtaking_model.x_for(0.5):+.2f} s/vuelta más rápido")

plot_logistic(attacks, "delta_pace", overtaking_model, bins=[-3, -1, -0.5, 0, 0.5, 1, 1.5, 2, 4],
              xlabel="Diferencia de ritmo teórico (s/vuelta, positiva si el atacante es más rápido)",
              title="México 2023-2025 · Diferencia de ritmo teórico");

## 4. Los tres argumentos del modelo de carrera

El modelo de carrera usa tres argumentos, que van en la configuración de la carrera (`RaceConfig`):

- `base_pace`: el ritmo base de cada piloto (s), con el neumático medio nuevo.
- `tyre_deg`: los coeficientes de cada compuesto, de la potencia más baja a la más alta: `[diferencia, degradación, ...]`. Con dos coeficientes es un modelo lineal, pero se pueden poner más para un polinomio.
- `overtaking_lr`: los coeficientes de la regresión logística, `[β0, β1, ...]`.

In [ ]:
overtaking_lr = [overtaking_model.intercept, overtaking_model.slope]

config = RaceConfig(base_pace=base_pace, tyre_deg=tyre_deg, overtaking_lr=overtaking_lr)

print("base_pace =", {driver: round(value, 3) for driver, value in sorted(base_pace.items(), key=lambda item: item[1])})
print("tyre_deg =", {name: [round(value, 3) for value in coefficients] for name, coefficients in tyre_deg.items()})
print("overtaking_lr =", [round(value, 3) for value in overtaking_lr])

Además de los tres modelos, la configuración (`RaceConfig`) recoge todo lo que no cambia entre simulaciones. Si no decimos otra cosa, son los valores de México 2025:

In [ ]:
config

- **71 vueltas** y una pérdida en boxes de **22,7 s**, medida con las paradas en bandera verde de México 2024 y 2025.
- La **parrilla** de México 2025. Lawson no está: abandonó en las primeras vueltas y no tenemos vueltas suyas para calcular su ritmo.
- `attack_gap` y `min_gap`: a menos de 1,5 s el coche de detrás ataca, y si no adelanta se queda a 0,4 s.

El modelo recorre la carrera **vuelta a vuelta**. En cada vuelta:

1. Cada coche tiene su **ritmo teórico**: ritmo base + neumático, y la pérdida en boxes si para en esa vuelta.
2. Los coches que paran van por el pit lane: no atacan ni les atacan.
3. Del primero al último, cada coche que empieza la vuelta a menos de 1,5 s del de delante **ataca**, es decir, intenta adelantar. La probabilidad de adelantar sale del modelo de adelantamiento, con la diferencia de ritmo teórico entre los dos.
4. Se ordenan los coches por tiempo acumulado.

En la primera vuelta no hay ataques: el modelo de adelantamiento no incluye las salidas.

La único variable aleatoria son los adelantamientos.

## 5. Una carrera: las estrategias reales de 2025

Empezamos con las estrategias que usaron los pilotos en 2025, cada una como una lista de stints `(compuesto, vueltas)`. Hemos simplificado algunas cosas:

- Todos los neumáticos empiezan nuevos (en la carrera real, algunos blandos tenían 3 vueltas).
- Alonso y Hülkenberg abandonaron: completamos su carrera con medios.

In [ ]:
strategies_2025 = {
    "NOR": [("S", 34), ("M", 37)],
    "LEC": [("S", 31), ("M", 40)],
    "HAM": [("S", 23), ("M", 24), ("S", 24)],
    "RUS": [("S", 25), ("M", 23), ("S", 23)],
    "VER": [("M", 37), ("S", 34)],
    "ANT": [("S", 22), ("M", 25), ("S", 24)],
    "PIA": [("S", 24), ("M", 23), ("S", 24)],
    "HAD": [("M", 34), ("S", 37)],
    "BEA": [("S", 24), ("M", 24), ("S", 23)],
    "TSU": [("M", 36), ("S", 35)],
    "OCO": [("S", 27), ("M", 44)],
    "SAI": [("M", 17), ("S", 29), ("S", 25)],
    "HUL": [("S", 25), ("M", 46)],
    "ALO": [("S", 20), ("M", 51)],
    "BOR": [("M", 37), ("S", 34)],
    "ALB": [("H", 41), ("S", 30)],
    "GAS": [("M", 33), ("S", 38)],
    "STR": [("S", 26), ("M", 17), ("S", 28)],
    "COL": [("H", 48), ("S", 23)],
}

result = simulate_race(config, strategies_2025, seed=42)
result.classification().round(1)

In [ ]:
plot_race_trace(result, title="México 2025 simulado · Diferencia con el ritmo medio");
plot_positions(result, title="México 2025 simulado · Posiciones");
plot_tyre_strategies(result, title="México 2025 simulado · Estrategias");

In [ ]:
plot_time_lost(result, title="México 2025 simulado · Tiempo perdido en tráfico");

In [ ]:
overtakes = result.overtakes()
print(f"Adelantamientos en pista: {len(overtakes)} de {int(result.laps['attack'].sum())} ataques")
overtakes.round(2).head(10)

## 6. La carrera de Leclerc

En esta carrera, Leclerc sale **segundo** y tiene el **cuarto mejor ritmo** de la parrilla, pero acaba **séptimo**. ¿Podía haberlo hecho mejor con otra estrategia?

Probamos varias opciones, todas saliendo con blando como en la carrera real, y dejamos a los demás pilotos con su estrategia.

In [ ]:
RIVALS = ["LEC", "NOR", "VER", "HAM", "RUS", "PIA", "ANT", "BEA"]

### Estrategia real: S - M31

Leclerc va segundo hasta que para, en la vuelta 31. Hamilton, Russell, Antonelli y Piastri han parado antes y le han hecho el undercut: sale de boxes séptimo. Recupera posiciones cuando ellos paran por segunda vez, pero en las últimas vueltas, con un medio de 40 vueltas, le alcanzan con blandos nuevos y le pasan.

In [ ]:
race = simulate_strategy(config, strategies_2025, driver="LEC", starting_tyre="S", stops=[("M", 31)])
print(f"Leclerc acaba {race.final_order.index('LEC') + 1}.º")
plot_race_trace(race, drivers=RIVALS, title="Leclerc · S - M31 (real)");

### Opción 1: duro en vez de medio (S - H31)

La principal limitación es la degradación al final de la carrera.

El duro es más lento al principio, pero degrada menos: en un stint de 40 vueltas termina siendo más rápido que el medio.

In [ ]:
race = simulate_strategy(config, strategies_2025, driver="LEC", starting_tyre="S", stops=[("H", 31)])
print(f"Leclerc acaba {race.final_order.index('LEC') + 1}.º")
plot_race_trace(race, drivers=RIVALS, title="Leclerc · S - H31 (evitar degradación)");

### Opción 2: evitar tráfico (S - H28)

Ahora, el problema principal es que LEC pierde mucho tiempo detrás de ANT, RUS y PIA, que le hacen un undercut.

Parar antes de que se materialice el undercut soluciona el problema.

In [ ]:
race = simulate_strategy(config, strategies_2025, driver="LEC", starting_tyre="S", stops=[("H", 28)])
print(f"Leclerc acaba {race.final_order.index('LEC') + 1}.º")
plot_race_trace(race, drivers=RIVALS, title="Leclerc · S - H28 (evitar tráfico)");

### Opción 3: hacer dos paradas (S - M28 - S50)

El último stint sigue siendo muy largo y la degradación casi le cuesta a LEC la tercera posición.

Podemos hacer más paradas y utilizar neumáticos más blandos.

In [ ]:
race = simulate_strategy(config, strategies_2025, driver="LEC", starting_tyre="S", stops=[("M", 28), ("S", 50)])
print(f"Leclerc acaba {race.final_order.index('LEC') + 1}.º")
plot_race_trace(race, drivers=RIVALS, title="Leclerc · S - M28 - S50 (dos paradas)");

### Opción 4: evitar el undercut de Hamilton (S - M22 - S46)

Parando antes hemos evitado el undercut de ANT, PIA y RUS, pero HAM tiene un ritmo parecido al de LEC y no podemos evitar que nos haga el undercut si paramos más tarde que él.

Nos podemos adelantar a su parada cambiando neumáticos **una vuelta antes** para evitar que nos haga el undercut.

In [ ]:
race = simulate_strategy(config, strategies_2025, driver="LEC", starting_tyre="S", stops=[("M", 22), ("S", 46)])
print(f"Leclerc acaba {race.final_order.index('LEC') + 1}.º")
plot_race_trace(race, drivers=RIVALS, title="Leclerc · S - M22 - S46 (evitar undercut de Hamilton)");

### Varias carreras

Igual que en el bloque de Fórmula E, para comparar estrategias corremos muchas carreras distintas y miramos la **distribución** de resultados.

In [ ]:
SIMULATIONS = 200

# Paradas de cada estrategia: todas salen con blando.
lec_stops = {
    "S - M31 (real)": [("M", 31)],
    "S - H31 (evitar degradación)": [("H", 31)],
    "S - H28 (evitar tráfico)": [("H", 28)],
    "S - M28 - S50 (dos paradas)": [("M", 28), ("S", 50)],
    "S - M22 - S46 (evitar undercut de Hamilton)": [("M", 22), ("S", 46)],
}

# Puntos de la F1 para los diez primeros.
POINTS = {1: 25, 2: 18, 3: 15, 4: 12, 5: 10, 6: 8, 7: 6, 8: 4, 9: 2, 10: 1}

positions, time_lost = {}, {}
for name, stops in lec_stops.items():
    positions[name], time_lost[name] = [], []
    for seed in range(SIMULATIONS):
        race = simulate_strategy(config, strategies_2025, driver="LEC", starting_tyre="S", stops=stops, seed=seed)
        positions[name].append(race.final_order.index("LEC") + 1)
        # Tiempo perdido detrás de otros coches: tiempo de cada vuelta menos su tiempo libre.
        lec = race.laps[race.laps["driver"] == "LEC"]
        time_lost[name].append((lec["lap_time"] - lec["free_lap_time"]).sum())

positions = pd.DataFrame(positions)
points = positions.apply(lambda column: column.map(POINTS).fillna(0))

summary = pd.DataFrame({
    "Posición media": positions.mean(),
    "Posición STD": positions.std(),
    "Puntos medios": points.mean(),
    "Puntos STD": points.std(),
    "Podio": (positions <= 3).mean(),
    "Mejor": positions.min(),
    "Peor": positions.max(),
    "Tiempo perdido en tráfico (s)": pd.DataFrame(time_lost).mean(),
})
with pd.option_context("display.float_format", "{:.2f}".format):
    display(summary.sort_values("Posición media"))


## 7. La carrera real de 2025

Para terminar, las mismas gráficas con la carrera real, construidas con los datos de OpenF1. Las bandas de color marcan las vueltas con Safety Car o Virtual Safety Car.

La carrera real tuvo cosas que el modelo no tiene: abandonos (Lawson, Alonso, Hülkenberg y Sainz), un Virtual Safety Car en las dos últimas vueltas, neumáticos usados en la salida y una penalización de 10 s a Hamilton. Por eso no la comparamos posición a posición con la simulada, pero sí podemos ver si se parecen el ritmo y las estrategias.

En la carrera real no sabemos el tiempo que habría hecho cada coche con pista libre, así que aquí no podemos calcular el tiempo perdido en tráfico.

In [ ]:
real = real_race_result(races[2025], overtaking_data[2025].positions, tlas)
real.classification().round(1)

In [ ]:
plot_race_trace(real, title="México 2025 real · Diferencia con el ritmo medio");
plot_positions(real, title="México 2025 real · Posiciones");
plot_tyre_strategies(real, title="México 2025 real · Estrategias");

## 8. La carrera de PIA

PIA tiene el tercer mejor ritmo base, y sin embargo queda 5.º tanto en la carrera real como en la mayoría de simulaciones. ¿Qué podrían haber hecho distinto desde el muro de McLaren?